# 01 · Load and look at the data

What is in this repo's data, step by step:

1. **Real ISS cabin air** for the demo: `public/data/cabin.json` (mission RR-7, hourly and daily)
2. **Real ISS cabin air** for all 10 missions: `scripts/snapshots/RR-*_hourly.json` (what the backtest uses)
3. **Simulated crew check-ins**: `public/data/crew.json`, only loaded here, explored in the next notebook

The CO₂ is the ISS telemetry NASA OSDR publishes alongside the Rodent Research payloads (`co2_iss`). No mouse data is used.
Units: the snapshots keep CO₂ in **ppm** as published; `cabin.json` converts it to **mmHg** (`ppm × 760 / 1e6`).

In [ ]:
import json
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd

# The notebook lives in notebooks/; the data is one level up.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "public" / "data"
SNAPSHOTS = ROOT / "scripts" / "snapshots"

CO2_LIMIT_MMHG = 3.0  # NASA-STD-3001 Vol 2 [V2 6004]: 1-hour average limit
PPM_TO_MMHG = 760 / 1e6

plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True, "grid.alpha": 0.3})
print(ROOT)

## 1. The demo cabin feed: `cabin.json` (real, RR-7)

Three parts: `meta` (where it came from), `daily` (one row per mission day) and `hourly`.

In [ ]:
cabin = json.loads((DATA / "cabin.json").read_text())
cabin["meta"]

In [ ]:
daily = pd.DataFrame(cabin["daily"])
daily["date"] = pd.to_datetime(daily["date"])
daily = daily.set_index("day")
daily

In [ ]:
hourly = pd.DataFrame(cabin["hourly"])
hourly["t"] = pd.to_datetime(hourly["t"])
print(f"{len(hourly)} hours, {hourly['t'].min()} -> {hourly['t'].max()}")
hourly.describe()

### Hourly CO₂ with the NASA limit

The big rise in late August 2018 is the one event the demo is built around (mission days 213-220).

In [ ]:
fig, ax = plt.subplots()
h = hourly.set_index("t")["co2"].asfreq("h")  # missing hours stay gaps
ax.plot(h.index, h.values, lw=0.8, label="hourly mean CO₂")
ax.axhline(CO2_LIMIT_MMHG, color="crimson", ls="--", label="NASA 1-h limit, 3 mmHg")
ax.set(title="ISS cabin CO₂ during RR-7 (real telemetry)", ylabel="CO₂ (mmHg)")
ax.legend()
plt.show()

### Daily view: CO₂, temperature and humidity per mission day

Each mission day is the 24 h before the 08:00 UTC check-in. `co2Mean` is empty when a day has fewer than 12 hours of data.
The shaded band marks the demo's golden-path days (212 → 223).

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
axes[0].plot(daily.index, daily["co2Mean"], marker="o", ms=3, label="24 h mean")
axes[0].plot(daily.index, daily["co2Max1h"], lw=0.8, alpha=0.6, label="max 1-h mean")
axes[0].axhline(CO2_LIMIT_MMHG, color="crimson", ls="--", label="3 mmHg limit")
axes[0].set_ylabel("CO₂ (mmHg)")
axes[0].legend(loc="upper left")
axes[1].plot(daily.index, daily["temp"], color="tab:orange")
axes[1].set_ylabel("Temp (°C)")
axes[2].plot(daily.index, daily["rh"], color="tab:green")
axes[2].set_ylabel("RH (%)")
axes[2].set_xlabel("mission day")
for ax in axes:
    ax.axvspan(212, 223, color="gold", alpha=0.15)
fig.suptitle("RR-7 cabin, daily (real)")
plt.tight_layout()
plt.show()

In [ ]:
# The golden-path window in numbers
daily.loc[210:224, ["date", "hours", "co2Mean", "co2Max1h", "temp", "rh"]]

## 2. All 10 missions: `scripts/snapshots/RR-*_hourly.json` (real)

Each snapshot has `meta` (source, checksum of the raw API response, how many samples were dropped) and `hourly` rows
`[hourUtc, co2Ppm, tempC, rhPct, nSamples]`. The backtest runs simulated crews over each of these traces.

In [ ]:
frames, info = [], []
for path in sorted(SNAPSHOTS.glob("RR-*_hourly.json")):
    snap = json.loads(path.read_text())
    m = snap["meta"]
    df = pd.DataFrame(snap["hourly"], columns=m["columns"])
    df["hourUtc"] = pd.to_datetime(df["hourUtc"])
    df["co2_mmhg"] = df["co2Ppm"] * PPM_TO_MMHG
    df["mission"] = m["mission"]
    frames.append(df)
    info.append({"mission": m["mission"], "title": m["missionTitle"], "start": df["hourUtc"].min().date(),
                 "end": df["hourUtc"].max().date(), "hours": len(df), **m["sampleCounts"]})

snapshots = pd.concat(frames, ignore_index=True)
missions = pd.DataFrame(info).set_index("mission").sort_values("start")
missions

In [ ]:
# CO2 per mission in mmHg: typical level and spread
snapshots.groupby("mission")["co2_mmhg"].describe().loc[missions.index].round(2)

In [ ]:
order = list(missions.index)
fig, axes = plt.subplots(len(order), 1, figsize=(11, 2 * len(order)), sharey=True)
for ax, mission in zip(axes, order):
    # missing hours stay gaps, so they are not drawn as data
    s = snapshots[snapshots["mission"] == mission].set_index("hourUtc")["co2_mmhg"].asfreq("h")
    ax.plot(s.index, s.values, lw=0.7)
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(ax.xaxis.get_major_locator()))
    ax.axhline(CO2_LIMIT_MMHG, color="crimson", ls="--", lw=0.8)
    ax.set_title(f"{mission}: {missions.loc[mission, 'title']}", fontsize=9, loc="left")
    ax.set_ylabel("mmHg")
fig.suptitle("Hourly ISS cabin CO₂ for each mission (red = 3 mmHg limit)")
plt.tight_layout()
plt.show()

## 3. Simulated crew: `crew.json` (only loaded here)

Four made-up people, one row per daily check-in. Explored in the next notebook.

In [ ]:
crew = json.loads((DATA / "crew.json").read_text())
checkins = pd.DataFrame(crew["checkIns"])
print(crew["meta"]["label"], "| seed", crew["meta"]["seed"], "| days", crew["meta"]["days"])
checkins.head(8)